# Phases 3-4 on a free Kaggle GPU: teacher labels and prompted baselines

1. **Teacher labels (phase 3):** a large open model labels a split via `label/teacher.py`.
2. **Baseline (b) (phase 4):** the small candidate student models, prompted with the same
   instructions and no fine-tuning, predict val. Scored here against the CFPB product; the
   other fields are scored on the laptop against the gold set and teacher labels.

Each model gets its own vLLM server, started and stopped in turn so the GPUs are free for the next.

**Before running**
1. Settings: *Accelerator* = **GPU T4 x2**, *Internet* = **on**.
2. Upload `data/processed/train.parquet` and `val.parquet` as a private Kaggle dataset and add it
   (*Add Input*). Set `SPLITS_DIR` below to its path.
3. To resume an earlier session: download `labels/*.jsonl` from its output, upload them as a
   dataset, add it, and set `PREVIOUS_LABELS`. Rows already labelled by the same model and
   prompt version are skipped.

Label **val first** (5k rows), score it against the gold set on the laptop, and only then run train.

In [ ]:
TEACHER = "Qwen/Qwen2.5-14B-Instruct-AWQ"   # fallback if it does not fit or is slow: Qwen/Qwen2.5-7B-Instruct-AWQ
TEACHER_GPUS = 2                            # tensor parallel over T4 x2; 1 for a single GPU
STUDENTS = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]  # phase 5 candidates
SPLIT = "val"                               # teacher split: val first, then "train"

RUN_TEACHER = True                          # set False to run only the baselines
RUN_BASELINES = True                        # quick (minutes); val only

SPLITS_DIR = "/kaggle/input/complaint-splits"
PREVIOUS_LABELS = None                      # e.g. "/kaggle/input/complaint-labels"
REPO = "https://github.com/bhzcode1/customer-s-complaint.git"

In [ ]:
!git clone -q {REPO} /kaggle/working/repo
!pip install -q vllm duckdb

In [ ]:
import os, shutil, glob, subprocess, time, urllib.request
os.chdir("/kaggle/working/repo")
for d in ("data/processed", "data/labels", "data/predictions"):
    os.makedirs(d, exist_ok=True)
# Find the split files wherever Kaggle mounted the dataset (the folder name follows the
# dataset's slug and can be nested), and stop here if they are missing.
for name in ("train.parquet", "val.parquet"):
    found = (glob.glob(f"{SPLITS_DIR}/**/{name}", recursive=True)
             or glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not found:
        print("Files under /kaggle/input:")
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                print("  ", os.path.join(root, f))
        raise FileNotFoundError(f"{name} not found: add the dataset with train/val.parquet (Add Input)")
    shutil.copy(found[0], "data/processed/")
    print(f"{name} <- {found[0]}")
if PREVIOUS_LABELS:
    for f in glob.glob(f"{PREVIOUS_LABELS}/**/*.jsonl", recursive=True):
        shutil.copy(f, "data/labels/")
print(os.listdir("data/processed"), os.listdir("data/labels"))

URL = "http://localhost:8000/v1"

def start_server(model, gpus=1):
    """vLLM in the background. T4 has no bfloat16, hence --dtype half."""
    log = open(f"/kaggle/working/vllm_{model.split('/')[-1]}.log", "w")
    server = subprocess.Popen(
        ["vllm", "serve", model, "--dtype", "half", "--max-model-len", "4096",
         "--tensor-parallel-size", str(gpus), "--gpu-memory-utilization", "0.92",
         "--served-model-name", model, "--port", "8000"],
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):  # download + load can take ~10 min for a large model
        if server.poll() is not None:
            break
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=5)
            print(f"{model}: server ready")
            return server
        except Exception:
            time.sleep(10)
    print(open(log.name).read()[-3000:])
    server.kill()
    raise RuntimeError(f"vLLM did not start for {model}")

def stop_server(server):
    server.terminate()
    try:
        server.wait(timeout=60)
    except subprocess.TimeoutExpired:
        server.kill()
    time.sleep(5)  # let the GPU memory be released before the next model

short = lambda model: model.split("/")[-1].lower()

## 1. Teacher labels

In [ ]:
if RUN_TEACHER:
    server = start_server(TEACHER, TEACHER_GPUS)
    # Smoke test: 20 rows into a separate file; look at them before the full run.
    !python label/teacher.py --split {SPLIT} --limit 20 --workers 16 --model {TEACHER} --base-url {URL} --out data/labels/smoke_{SPLIT}.jsonl
    !head -c 2000 data/labels/smoke_{SPLIT}.jsonl

In [ ]:
if RUN_TEACHER:
    # Full run. Prints rows/s every 50 rows; safe to stop and resume (see step 3 above).
    !python label/teacher.py --split {SPLIT} --workers 32 --model {TEACHER} --base-url {URL}
    stop_server(server)

## 2. Baseline (b): small models, prompted, no fine-tuning

In [ ]:
if RUN_BASELINES:
    for model in STUDENTS:
        server = start_server(model, 1)
        out = f"data/predictions/base_{short(model)}_val.jsonl"
        !python label/teacher.py --split val --workers 64 --model {model} --base-url {URL} --out {out}
        stop_server(server)
        !python eval/evaluate.py {out}

## 3. Save outputs (download from the Output tab, or *Save Version*)

In [ ]:
for src, dst in (("data/labels", "/kaggle/working/labels"), ("data/predictions", "/kaggle/working/predictions")):
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f"{src}/*.jsonl"):
        shutil.copy(f, dst)
    print(dst, os.listdir(dst))